# Fine-tuning Llama-PLLuM-8B-chat — QLoRA

- **Model:** `CYFRAGOVPL/Llama-PLLuM-8B-chat`  
- **Licencja:** Llama 3.1 (open, dozwolony użytek komercyjny)  
- **Metoda:** QLoRA (4-bit) via Unsloth  
- **Kaggle**: `GPU T4 x 2`
- **Czas treningu:** ~5 min (1 epoka, mały dataset)

### Przed startem:
- `Kaggle` → `Settings` → `Accelerator` → **T4 GPU x 2**

## 1. Instalacja

In [1]:
%%capture
!pip install unsloth
!pip install trl accelerate peft datasets bitsandbytes

## 2. Sprawdzenie GPU

In [2]:
!nvidia-smi

Fri Apr 17 15:18:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Ładowanie modelu + QLoRA

> **Uwaga:** Pobieranie modelu 8B zajmie ~2–3 min. VRAM po załadowaniu: ok. 6–7 GB.

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="CYFRAGOVPL/Llama-PLLuM-8B-chat",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Adaptery LoRA — target_modules identyczne jak dla Llama 3.1
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    # lora_dropout=0.05,
    lora_dropout=0.00,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"VRAM zajęte: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Dataset

Przykładowe dane w formacie `{instruction, input, output}`

> **⚠️ Ważne:** PLLuM używa formatu **Llama 3** (`<|start_header_id|>`), NIE ChatML (`<|im_start|>`).
> To kluczowa różnica względem SmolLM2!

In [ ]:
from datasets import Dataset

# ===== DANE =====
data = [
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest machine learning?",
        "output": "Machine learning to dziedzina AI, w której modele uczą się na danych bez jawnego programowania."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest fine-tuning?",
        "output": "Fine-tuning to dostosowanie pretrenowanego modelu do konkretnego zadania poprzez dalsze trenowanie na małym zbiorze danych."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest LoRA?",
        "output": "LoRA (Low-Rank Adaptation) to metoda fine-tuningu, która trenuje tylko małe adaptery zamiast całego modelu — oszczędza VRAM i czas."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest tokenizer?",
        "output": "Tokenizer to algorytm zamieniający tekst na liczby (tokeny) zrozumiałe dla modelu językowego."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest GPU?",
        "output": "GPU to procesor graficzny — ma tysiące małych rdzeni, idealnych do równoległych obliczeń macierzowych potrzebnych w AI."
    },
]
# ===== KONIEC DANYCH =====

# Format chat template — LLAMA 3 (nie ChatML!)
# PLLuM używa: <|begin_of_text|><|start_header_id|>role<|end_header_id|>\n\ntreść<|eot_id|>
def format_prompt(example):
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{example['instruction']}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['input']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}

dataset = Dataset.from_list(data).map(format_prompt)
print(f"Przykładów w datasecie: {len(dataset)}")
print("\nPrzykład:")
print(dataset[0]["text"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Przykładów w datasecie: 5

Przykład:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Odpowiedz krótko po polsku.<|eot_id|><|start_header_id|>user<|end_header_id|>

Co to jest machine learning?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Machine learning to dziedzina AI, w której modele uczą się na danych bez jawnego programowania.<|eot_id|>


## 5. Trening

> **Uwaga dot. batch_size:** Zmniejszamy z 2 → 1 (model 8B zajmuje więcej VRAM niż 1.7B).  
> Gradient accumulation 8 kompensuje — efektywny batch = 8 (taki sam jak wcześniej).

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig( 
        dataset_text_field="text",
        max_seq_length=512, 
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="./output",
        optim="adamw_8bit",
        save_strategy="no",
        report_to="none",
    ),
)

print("Start treningu...")
trainer.train()
print("✅ Trening zakończony!")

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.
[datasets.arrow_dataset|WARNING]num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Start treningu...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,220,672 (0.52% trained)


Step,Training Loss
1,6.210471


✅ Trening zakończony!


## 6. Test modelu po fine-tuningu

In [ ]:
FastLanguageModel.for_inference(model)

def zapytaj(pytanie, instrukcja="Odpowiedz krótko po polsku."):
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{instrukcja}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{pytanie}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        return response.split("assistant")[-1].strip()
    return response.strip()

# Test
print(zapytaj("Co to jest LoRA?"))
print("---")
print(zapytaj("Wyjaśnij czym jest PLLuM."))

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

LoRA (Learning to Rank for Attention) to sieć neuronowa, która została zaprojektowana do optymalizacji uwagi w modelach transformer. Została zaproponowana w 2022 roku przez Google AI Research.
---
PLLuM to skrót od Polish Large Language Model. Jest to zaawansowany model językowy, stworzony przez konsorcjum polskich instytucji naukowych, w którego skład wchodzą: Politechnika Wrocławska (lider projektu), Naukowa i Akademicka Sieć Komputerowa – Państwowy Instytut Badawczy, Instytut Podstaw Informatyki Polskiej Akademii Nauk, Ośrodek Przetwarzania Informacji – Państwowy Instytut Badawczy, Uniwersytet Łódzki i Instytut Slawistyki Polskiej Akademii Nauk. Model został upubliczniony w grudniu 2024 r.
